In [19]:
import pandas as pd
import numpy as np
from evidently import Report
import warnings
from evidently.presets import DataDriftPreset
from evidently.presets import DataDriftPreset

warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv('../data/processed/train_processed.csv')

# Simulate: first 80% = reference (training), last 20% = current (production)
split = int(len(df) * 0.8)
reference = df.iloc[:split].copy()
current   = df.iloc[split:].copy()

# Rename target for Evidently
reference = reference.rename(columns={'isFraud': 'target'})
current   = current.rename(columns={'isFraud': 'target'})

print(f"Reference (training) : {reference.shape}")
print(f"Current (production) : {current.shape}")

Reference (training) : (472432, 435)
Current (production) : (118108, 435)


In [18]:
# Use top 20 features only (faster)
top_features = [
    'TransactionAmt', 'TransactionAmt_log', 'TransactionDT',
    'card1', 'card2', 'card3', 'card5',
    'hour', 'day', 'week',
    'card1_amt_mean', 'card1_amt_std', 'card1_freq',
    'amt_deviation', 'target'
]

ref_sample = reference[top_features].sample(n=10000, random_state=42)
cur_sample = current[top_features].sample(n=5000, random_state=42)

report = Report(metrics=[DataDriftPreset()])

my_eval = report.run(reference_data=ref_sample, current_data=cur_sample)

import os
os.makedirs('../models', exist_ok=True)
my_eval.save_html('../models/drift_report.html')